# Submission 2  ADBS 2026
**Tomilin Eugene 12533692**

## Exercise 1 — Distributed Joins

### Data sizes
#### record & cluster placement
| Relation | Records | Record size | site  |
| -------- | ------: | ----------: | ----: |
| Authors  |   4,000 |       450 B |   1   |
| Books    |  50,000 |       400 B |   2   |
| Libraries|  10,000 |       750 B |   3   |

#### attribute
| Attribute        | Size |
| ---------------- | ---: |
| Authors.authid   | 12 B |
| Authors.name     | 58 B |
| Books.author     | 10 B |
| Books.title      | 30 B |
| Books.library    | 10 B |
| Libraries.libid  | 10 B |

#### Calculus
  
_projection record count_ is A ⋈ B = A × B × selectivity \
_projected tuple size_ for Books.title + Authors.name is 30 + 58 B \
_total size_ then 4000 × 50000 × 1/9500 × (30 + 58) ~ 1852576 B \
I intentionally showing it each time in expression, so its clear where value is taken from.

### Strategy a.1 — Send Both Tables to Site 4

```mermaid
graph LR
    S1["Site 1<br/>Authors<br/>4000 * 450B = 1.8 MB"]
    S2["Site 2<br/>Books<br/>50000 * 400B = 20 MB"]
    S4["Site 4<br/>Join + Result"]

    S1 -- "1.8 MB" --> S4
    S2 -- "20 MB" --> S4
```
a_1 = (4000 × 450) + (50000 × 400) = 1800000 + 20000000 = 21800000 B \
*Total transfer: 21.8 MB*

### Strategy a.2 — Send Authors to site 2, join there and send the result to site 4

```mermaid
graph RL
    S1["Site 1<br/>Authors<br/>1.8 MB"]
    S2["Site 2<br/>Books"]
    S4["Site 4<br/>Final Result"]

    S1 -- "Authors: 1.8 MB" --> S2
    S2 -- "Result: 1.85 MB" --> S4
```
a_2 = (4000 × 450) + (4000 × 50000 × 1/9500 × (58+30)) = 1800000 + 1852576 = 3652576 B \
*Total transfer: 3.65 MB*

## Strategy a.3 — Send Books to site 1, join there and send the result to site 4.

```mermaid
graph LR
    S1["Site 1<br/>Authors"]
    S2["Site 2<br/>Books<br/>20 MB"]
    S4["Site 4<br/>Final Result"]

    S2 -- "Books: 20 MB" --> S1
    S1 -- "Result: 1.85 MB" --> S4
```
a_3 = (50000 × 400) + (4000 × 50000 × 1/9500 × (58+30)) = 20000000 + 1852576 = 21852576 B \
*Total transfer: 21.85 MB*

## Strategy a.4 — Send only the join attributes of Authors to site 2, semi join with Books, and send the result back to site 1 to compute the full join. Finally transfer the full join to site 4.

```mermaid
graph LR
    S1["Site 1<br/>Authors"]
    S2["Site 2<br/>Books"]
    S4["Site 4<br/>Final Result"]

    S1 -- "authid only<br/>48 KB" --> S2
    S2 -- "matching Books rows<br/>8.42 MB" --> S1
    S1 -- "Result: 1.85 MB" --> S4
```

processing : \
Step 1: Send authid only: Tuple size = 12 B 4000×12=48,000 \
Step 2: Estimate semi-join result size. \
  Hint says: take min , semijoin returns matching Books tuples \
    join result ~ 21052, and Books has 50000 rows so  min(21052,50000)=21052 \
Step 3: Send matching Books rows back to 1 . Full Books tuples: 21052×400 \
Step 4: compute full join on site 1 \
Step 5: Send final result to site 4 

a_4 = (4000 × 12) + (min(4000 × 50000 × 1/9500, 50000) × 400) + (4000 × 50000 × 1/9500 × 88)= 48000 + 8420800 + 1852576 = 10321376 B \
*Total transfer: 10.32 MB*

## Strategy a.5 —The semi-join strategy in the opposite direction.

```mermaid
graph LR
    S1["Site 1<br/>Authors"]
    S2["Site 2<br/>Books"]
    S4["Site 4<br/>Final Result"]

    S2 -- "author ids only<br/>500 KB" --> S1
    S1 -- "matching Authors rows<br/>1.8 MB" --> S2
    S2 -- "Result: 1.85 MB" --> S4
```

a_5 = (50000 × 10) + (min(4000 × 50000 × 1/9500, 4000) × 450) + (4000 × 50000 × 1/9500 × 88)= 500000 + 1800000 + 1852576 = 4152576 B
 *Total transfer: 4.15 MB*

#### Best strategy conclusion  

| Case | Strategy | Cost MB | Best |
|---|---|---:|---|
| 1 | Send both tables to Site 4 | 21.80 MB | |
| 2 | Send Authors to Site 2 | 3.65 MB | v |
| 3 | Send Books to Site 1 | 21.85 MB | |
| 4 | Semi-join Authors to Books | 10.32 MB | |
| 5 | Semi-join Books to Authors | 4.15 MB | |

### a (star) Adapt the strategies that you always project only the necessary columns as early as possible
Paths remain the same, but only columns needed later are transferred not whole sets.
From Authors: authid (for join) , name (for final output).
From Books: author (for join) title (for final output). 

Data volumes decrease, effectively 
- Authors tuples become 70 B instead of 450 B 
- Books tuples become 40 B instead of 400 B 

Calculus remains alike, final table becomes 

| Case | Strategy | Cost MB | Best |
|---|---|---:|---|
| 1 | Send both projected tables to Site 4 | 2.28 MB | |
| 2 | Send projected Authors to Site 2 | 2.13 MB | v |
| 3 | Send projected Books to Site 1 | 3.85 MB | |
| 4 | Semi-join Authors to Books with projection | 2.74 MB | |
| 5 | Semi-join Books to Authors with projection | 2.63 MB | |

### b , generalize a (star) calc optimal for more then 2 relations.
Since I've already calculated options for Authors ⋈ Books in a(star) and given same cluster but extra Libraries join ( Authors ⋈ Books ⋈ Libraries ).
I'll continue with best intermediate result, no need to recalculate all because:
 - first join gives the most size to total trasnfer cost,
 - case 2 from (a*) already minimal,
 - strategies moving large Books/Libraries tables cannot win against projections
 - any strategy that moves large final result multiple times will exceed optimal path

Thus optimal strategy is:

```mermaid
graph LR
subgraph SITE1["Site 1"]
    S1["Authors<br/>(authid,name)<br/>70 B tuple"]
end
subgraph SITE2["Site 2"]
    S2["Books<br/>(author,title,library)<br/>50 B tuple"]
    J1["Local Join<br/>Authors ⋈ Books"]
end
subgraph SITE3["Site 3"]
    S3["Libraries<br/>10000 rows<br/>750 B tuple"]
end
subgraph SITE4["Site 4"]
    J2["Join<br/>Intermediate ⋈ Libraries"]
    R["Final Result"]
end
S1 -->|280 KB| J1
J1 -->|Intermediate<br/>1.43 MB| J2
S3 -->|Libraries<br/>7.5 MB| J2
J2 --> R
```

This path:
 - move smallest data possible,
 - never move Books table (largest raw relation),
 - compute intermediate at Site 2, then join with Libraries at Site 4.

#### calculus
Selectivity is the same for (Authors ⋈ Books) = 1/9500 and (outer join with Libraries) = 1/4000 so :

**Step 1:** Send projected Authors (authid, name) from Site 1 to Site 2 \
   projection size = 12 + 58 = 70 B ; data moved = 4000 × 70 = 280000 = *280 KB*

**Step 2:** Compute intermediate at Site 2 (Authors ⋈ Books) \
   rows_ab = 4000 × 50000 × 1/9500 ~ 21052 \
   projection size (name, library) = 58 + 10 = 68 B \
   total_ab = 21052 × 68 = 1431536 ~ *1.43 MB*

**Step 3:** Send intermediate from Site 2 → Site 4 \
   transfer = *1.43 MB*

**Step 4:** Send Libraries from Site 3 to Site 4 \
   transfer = 10000 × 750 = 7500000 = *7.5 MB*

**Step 5:** Final join at Site 4 (result already there)

**Total communication cost:** 280 KB + 1.43 MB + 7.5 MB = **9.21 MB**




## Exercise 2 Map reduce 
### Subtask 2.a : Show the intermediate results of each map phase and reduce phase of the three stages

The task is to illustrate map reduce sequence, I'm using yaml as compact format to show I/O as structure (input + output) with extra tags to illustrate grouping, filtering and calculations.

### graph from task 
```yaml
Graph:
  users: [u1,u2,u3,u4]
  items: [i1,i2,i3,i4,i5]
  edges:
    u1: [i1,i2]
    u2: [i1,i2,i3]
    u3: [i2,i4]
    u4: [i3,i5]
```

Execution sequence and IO
```yaml
Mapper1:
  rule: reverse edge direction item,user
  input: [[u1,i1],[u1,i2],[u2,i1],[u2,i2],[u2,i3],[u3,i2],[u3,i4],[u4,i3],[u4,i5]]
  output: [[i1,u1],[i2,u1],[i1,u2],[i2,u2],[i3,u2],[i2,u3],[i4,u3],[i3,u4],[i5,u4]]

Reducer1:
  rule: group users by item
  input: [[i1,u1],[i2,u1],[i1,u2],[i2,u2],[i3,u2],[i2,u3],[i4,u3],[i3,u4],[i5,u4]]
  groups:
    i1: [u1,u2]
    i2: [u1,u2,u3]
    i3: [u2,u4]
    i4: [u3]
    i5: [u4]
  output: [[i1,[u1,u2]],[i2,[u1,u2,u3]],[i3,[u2,u4]],[i4,[u3]],[i5,[u4]]]

Mapper2:
  rule: for each item generate all user pairs sharing that item
  input: [[i1,[u1,u2]],[i2,[u1,u2,u3]],[i3,[u2,u4]],[i4,[u3]],[i5,[u4]]]
  output:
    i1: [[[u1,u2],1]]
    i2: [[[u1,u2],1],[[u1,u3],1],[[u2,u3],1]]
    i3: [[[u2,u4],1]]
    i4: []
    i5: []

Reducer2:
  rule: sum similarity counts for every user pair
  input: [[[u1,u2],1],[[u1,u2],1],[[u1,u3],1],[[u2,u3],1],[[u2,u4],1]]
  groups:
    [u1,u2]: [1,1]
    [u1,u3]: [1]
    [u2,u3]: [1]
    [u2,u4]: [1]
  output: [[[u1,u2],2],[[u1,u3],1],[[u2,u3],1],[[u2,u4],1]]

Mapper3:
  rule: duplicate similarity edges for both users
  input: [[[u1,u2],2],[[u1,u3],1],[[u2,u3],1],[[u2,u4],1]]
  output:
    [[u1,u2],2]: [[u1,[u2,2]],[u2,[u1,2]]]
    [[u1,u3],1]: [[u1,[u3,1]],[u3,[u1,1]]]
    [[u2,u3],1]: [[u2,[u3,1]],[u3,[u2,1]]]
    [[u2,u4],1]: [[u2,[u4,1]],[u4,[u2,1]]]

Reducer3:
  rule: keep only top k most similar users for each user
  k: 2
  groups:
    u1: [[u2,2],[u3,1]]
    u2: [[u1,2],[u3,1],[u4,1]]
    u3: [[u1,1],[u2,1]]
    u4: [[u2,1]]
  topK:
    u1: [[u2,2],[u3,1]]
    u2: [[u1,2],[u3,1]]
    u3: [[u1,1],[u2,1]]
    u4: [[u2,1]]
  output: [[[u1,u2],2],[[u1,u3],1],[[u2,u1],2],[[u2,u3],1],[[u3,u1],1],[[u3,u2],1],[[u4,u2],1]]
```
### Subtask 2.b : Analyze cost measures for each stage of the MapReduce

#### Stage 1 (Communication = |E|, Replication = 1, Reducer size = max deg(v))

**Mapper1** processes **9** edges. Each edge produces exactly one output tuple. \
_Communication cost_: 9 × 1 = 9 \
Mapper input size = 9 , Mapper output size = 9 \
_Replication rate_: 9 / 9 = **1**

**Reducer1** groups users by item. \
Item degrees: deg(i1)=2, deg(i2)=3, deg(i3)=2, deg(i4)=1, deg(i5)=1 \
_Reducer size_ is largest reducer group = max(2,3,2,1,1) = **3**

#### Stage 2 [Communication = sum C(deg(v),2), Replication = [sum C(deg(v),2)] / |E|, Reducer size = max |CI(u1,u2)|]

For every item with degree d: generated pairs = d(d-1)/2 \
Item i1:2×1/2 = 1\
Item i2:3×2/2 = 3\
Item i3:2×1/2 = 1\
Item i4:1×0/2 = 0\
Item i5:1×0/2 = 0\
Total _communication cost_:  1 + 3 + 1 + 0 + 0 = **5**

**Mapper2** input records = 5 \
**Mapper2** output records = 5 \
_Replication rate_: 5 / 5 = **1**

**Reducer2** groups by user pair. \
Shared items: \
CI(u1,u2)=2, CI(u1,u3)=1, CI(u2,u3)=1, CI(u2,u4)=1 \
_Reducer size_ is largest reducer group which is max(2,1,1,1) = **2**

#### Stage 3 [Stage 3: Communication = 2 * |pairs|, Replication = 2, Reducer size = max |S(u)|]

**Reducer2** produces 4 similarity pairs. \
**Mapper3** duplicates every pair for both users. \
_Communication cost_: 4 × 2 = **8**

**Mapper3** input size = 4 \
**Mapper3** output size = 8 \
_Replication rate_: 8 / 4 = **2**

**Reducer3** groups by user. \
Number of similar users: S(u1)=2, S(u2)=3, S(u3)=2, S(u4)=1 \
_Reducer size_: max(2,3,2,1) = **3**

## Exercise 3 (MapReduce Algorithms) 

## Subtask (a)
#### Original Expression

```text
F1(X,Y,Z) = γ_b SUM(a) ( σ_(a>b)(X) − π_(a,b)(Y ⋈ Z) )
```
**Where**
```text
sets  
 X(a,b) Y(a,b,c) Z(b,c,d)

operations
⋈        join
π        projection
σ        selection/filter
γ        grouping/aggregation
−        set difference

σ_(a>b)(X)  *Take relation X and keep only rows where a>b.*
Y ⋈ Z  *Join Y and Z using their shared attributes b and c.*
π_(a,b)(...)  *From this join keep only columns (a,b).*
A − B *Remove all tuples from filtered X that also appear in the projected join result.*
γ_b SUM(a)  *group remaining tuples by b and compute SUM(a) for each group.*

### SQL equivalent
It's a hint for me to count and split on map reduce sequence
1 expression member -> 1 function (except FROM)
```sql
SELECT b,                    -- Reducer3 grouping key
       SUM(a)                -- Reducer3 aggregation
FROM (
        SELECT a,b           -- Mapper2 emits ((a,b),"X")
        FROM X
        WHERE a>b            -- Mapper2 filter σ(a>b)
          EXCEPT             -- Reducer2 set difference
        SELECT Y.a,Y.b       -- Mapper2 emits ((a,b),"R")
        FROM Y
        JOIN Z               -- Reducer1 reconstructs Y join Z
          ON Y.b = Z.b       -- Mapper1 join key part 1
         AND Y.c = Z.c       -- Mapper1 join key part 2

     ) T
GROUP BY b;                  -- Reducer3 computes γ_b SUM(a)
```
### Execution plan
```text
1. Filter X using a>b.
2. Join Y and Z on (b,c).
3. Project only (a,b).
4. Compute set difference.
5. Group by b.
6. Compute SUM(a).
```
### Map reduce sequence

```yaml
Mapper1:
  input: [Y(a,b,c),Z(b,c,d)]
  output: [((b,c),("Y",a,b)),((b,c),("Z"))]
  description: \
    For every tuple from Y emit key (b,c) tagged with "Y". \
    For every tuple from Z emit key (b,c) tagged with "Z".

Reducer1:
  input: {key:(b,c),values:["Y","Z"]}
  output: [(a,b)]
  description: \
    If matching tuples from Y and Z exist for the same key (b,c), emit (a,b).

Mapper2:
  input: [X(a,b),R(a,b)]
  output: [((a,b),"X"),((a,b),"R")]
  description: \
    For tuples from X satisfying a>b emit ((a,b),"X"). \
    For tuples from R emit ((a,b),"R").

Reducer2:
  input: {key:(a,b),values:["X","R"]}
  output: [(a,b)]
  description: \
    Emit (a,b) only if tuple exists in X and does not exist in R.

Mapper3:
  input: [(a,b)]
  output: [(b,a)]
  description: \
    Emit b as key and a as value.

Reducer3:
  input: {key:b,values:[a]}
  output: [(b,SUM(a))]
  description: \
    Compute SUM(a) for every group b.
```
## Subtask (b)

#### Original Expression

```text
M'ij = ( Mij - min(Mi) ) / ( max(Mi) - min(Mi) )
```
*Where*
```text
M        original matrix
M'       normalized matrix
i        row index
j        column index
Mij      element at row i and column j
min(Mi)  minimum value in row i 
max(Mi)  maximum value in row i
```
**implied algo**\
For every row independently:
1. Find the minimum value.
2. Find the maximum value.
3. Normalize every element using: (value-min)/(max-min).
This transforms all M matrix values into range [0,1].

Note: If all values in the row are equal, division by zero would occur at *max(Mi) - min(Mi)*, therefore all normalized values become 0.

### SQL equivalent

```sql
WITH RowStats AS (
    SELECT i,                   -- Reducer1 grouping key
           MIN(val) AS minv,    -- Reducer1 minimum aggregation
           MAX(val) AS maxv     -- Reducer1 maximum aggregation
    FROM M
    GROUP BY i  )

SELECT M.i,
       M.j,
       CASE
           WHEN S.maxv = S.minv THEN 0 -- handle div by 0
           ELSE (M.val - S.minv) / (S.maxv - S.minv)
       END                       -- Reducer2 normalization
FROM M
JOIN RowStats S ON M.i = S.i;    -- Reducer1 min/max reused
```

### Execution plan

1. Group matrix values by row i.
2. Compute MIN and MAX for every row.
3. Join row min/max back with original matrix.
4. Normalize every matrix element, handling division-by-zero.

### Map reduce sequence

```yaml
Mapper1:
  input: [M(i,j,val)]
  output: [(i,val)]
  description: \
    Emit row index i as key and matrix value val as value.

Reducer1:
  input: {key:i,values:[val]}
  output: [(i,min,max)]
  description: \
    Compute MIN(val) and MAX(val) for every row i.

Mapper2:
  input: [M(i,j,val),(i,min,max)]
  output: [(i,(j,val,min,max))]
  description: \
    Emit row i as key and attach matrix value together with row min/max.

Reducer2:
  input: {key:i,values:[(j,val,min,max)]}
  output: [(i,j,normalized_val)]
  description: \
    If max=min emit 0. \
    Otherwise compute (val-min)/(max-min).
```
